# Aquaplanet with a customized initial condition

Everything about this run is the shipped `aquaplanet-slab` configuration except one field of the initial carry.

In [ ]:
from pathlib import Path

import jax.numpy as jnp
from hydra import compose, initialize_config_module

import jem.config  # noqa: F401  -- registers the ${jem_data:}/${jcm_data:} resolvers
from jem import plot, replace_field, run_chunked
from jem import runners

output_dir = (Path("output") / "01-02_customized_initial_condition").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

## Build the shipped model

In [ ]:
with initialize_config_module(config_module="jem.config", version_base="1.3"):
    cfg = compose(config_name="config", overrides=["+configuration=aquaplanet-slab"])
coupler = runners.build_coupler(cfg)
coupler

## Change one initial condition

`Coupler.initialize()` returns a `CoupledCarry`; `replace_field` returns a new one with a single `"component.section.field"` replaced, so the original is untouched.

In [ ]:
carry = coupler.initialize()
grid = coupler.components["ocn"].grid
bump = 5 * jnp.sin(2 * grid.longitude_radian) * jnp.cos(grid.latitude_radian) ** 3
sst = carry.components["ocn"]["state"].sea_surface_temperature
carry = replace_field(carry, "ocn.state.sea_surface_temperature", sst + bump)

In [ ]:
import numpy as np
import xarray as xr

# 2-D lat/lon in degrees, straight from the grid -- map_plot handles
# 2-D auxiliary coordinates the same way it does the displaced-pole
# ocean's, so this needs no separate 1-D-vs-2-D case here either.
perturbed_sst = xr.DataArray(
    sst + bump - 273.15,
    dims=("lon_index", "lat_index"),
    coords={
        "lon": (("lon_index", "lat_index"), np.degrees(grid.longitude_radian)),
        "lat": (("lon_index", "lat_index"), np.degrees(grid.latitude_radian)),
    },
    name="sea_surface_temperature",
)
plot.map_plot(perturbed_sst, title="Perturbed initial sea surface temperature [°C]")

## Run it

In [ ]:
result = run_chunked(coupler, total_time=30, chunk=30,
   initial_carry=carry, output_dir=str(output_dir), subsample=3,
   checkpoint_path=None)

In [ ]:
import matplotlib.pyplot as plt

ocn = plot.open_output(output_dir, "ocn")

fig, ax = plt.subplots()
sst_final = ocn["sea_surface_temperature"].isel(time=-1) - 273.15
plot.map_plot(sst_final, ax=ax, title="Final sea surface temperature [°C]")

fig, ax = plt.subplots()
plot.area_mean(ocn["sea_surface_temperature"]).plot(ax=ax)
ax.set_ylabel("Area-mean SST [K]")